# professor에게 듣는 연구 한마디

<a href="https://colab.research.google.com/github/jumincho/AI-JBNU-2026/blob/main/notebooks/professor_voice_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

짧은 참조 음성(`voice_sample.mp3`, 약 7초)의 목소리를 [OmniVoice](https://github.com/k2-fsa/OmniVoice)로 복제해서, 입력한 문장을 그 목소리로 읽어 줍니다.

### Colab만으로 하기 (VSCode 없이)

1. 메뉴 **런타임 > 런타임 유형 변경**에서 하드웨어 가속기가 **T4 GPU**인지 확인하고 저장합니다.
2. 메뉴 **런타임 > 모두 실행**을 누릅니다. Google이 만든 노트북이 아니라는 경고가 나오면 **계속 실행**(Run anyway)을 누릅니다.
3. 처음에는 설치와 모델 다운로드 때문에 5~10분 정도 걸립니다. 끝나면 **7번 셀**에서 기본 문장을 들을 수 있습니다.
4. **6번 셀** 입력 칸에 원하는 문장을 적고, **6번과 7번 셀만** 다시 실행합니다. 셀 왼쪽의 실행 버튼을 누르면 됩니다.

### VSCode에서 Codex나 Claude Code와 함께 하기

1. 이 노트북을 VSCode에서 열고 커널을 **Colab > New Colab Server > GPU**로 연결합니다. 자세한 순서는 [저장소 README](https://github.com/jumincho/AI-JBNU-2026)에 있습니다.
2. 에이전트 대화창에서 professor 스킬로 음성용 답변을 받습니다.
   - Codex: `$professor 음성용으로 2~4문장만 답해 주세요. 교수님, 연구가 너무 하기 싫어요.`
   - Claude Code: `/professor 음성용으로 2~4문장만 답해 주세요. 교수님, 연구가 너무 하기 싫어요.`
3. 1번부터 5번 셀을 실행하고, 받은 답변을 6번 셀에 붙여넣은 뒤 6번과 7번 셀을 실행합니다.

언어, 스타일, 속도 같은 생성 옵션은 건드리지 않고 기본값으로 합성합니다. 그대로 두는 편이 가장 자연스럽게 들립니다.

## 1. GPU 확인하기

지금 연결된 컴퓨터에 GPU가 있는지 확인합니다. GPU 이름이 나오면 성공입니다.

GPU가 없다는 오류가 나오면 Colab에서는 **런타임 > 런타임 유형 변경 > T4 GPU**를 고르고 저장한 뒤 **런타임 > 모두 실행**을 다시 누릅니다.
VSCode에서는 오른쪽 위 커널 선택에서 **Colab > New Colab Server**를 고르고 GPU를 선택합니다.

In [ ]:
#@title 1. GPU 확인
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 연결되지 않았습니다. Colab: 런타임 > 런타임 유형 변경 > T4 GPU를 고르고 다시 실행하세요. "
        "VSCode: 커널을 Colab > New Colab Server > GPU로 연결하세요."
    )
print("GPU:", torch.cuda.get_device_name(0))

## 2. OmniVoice 설치하기

처음 한 번만 설치합니다. 2~3분 정도 걸리니 끝날 때까지 기다려 주세요.
Colab에 이미 있는 PyTorch는 그대로 두고, 수업 중에 바뀌지 않도록 OmniVoice 버전을 0.2.1로 고정합니다.

In [ ]:
#@title 2. OmniVoice 설치
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    installed = version("omnivoice")
except PackageNotFoundError:
    installed = None

if installed == "0.2.1":
    print("OmniVoice가 이미 설치되어 있습니다.")
else:
    # 설치 중에 PyTorch가 다른 버전으로 바뀌지 않도록 지금 버전을 고정합니다.
    pins = []
    for package in ("torch", "torchaudio"):
        try:
            pins.append(f"{package}=={version(package)}")
        except PackageNotFoundError:
            pass
    pin_file = Path(tempfile.gettempdir()) / "torch-pins.txt"
    pin_file.write_text("\n".join(pins) + "\n")

    print("OmniVoice를 설치하는 중입니다. 2~3분 걸립니다.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "omnivoice==0.2.1", "-c", str(pin_file)],
        check=True,
    )
    print("설치를 마쳤습니다.")

## 3. 참조 음성 준비하기

저장소에 있는 `voice_sample.mp3`(약 7초)를 가져와서 들어 봅니다. OmniVoice는 3~10초 길이의 참조 음성을 권장합니다.

다른 목소리를 쓰고 싶다면 본인의 목소리나 동의를 받은 목소리만 사용하세요.
파일을 올린 뒤 아래 `REFERENCE_AUDIO`에 파일 이름을 적습니다.
Colab은 왼쪽 폴더 아이콘에서 업로드하고, VSCode는 탐색기에서 파일을 우클릭한 뒤 **Upload to Colab**을 누릅니다.

In [ ]:
#@title 3. 참조 음성 준비
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import Audio, display

REFERENCE_AUDIO = "voice_sample.mp3"  # 다른 음성을 쓰려면 올린 파일 이름으로 바꿉니다.
SAMPLE_URL = "https://raw.githubusercontent.com/jumincho/AI-JBNU-2026/main/voice_sample.mp3"

reference = Path(REFERENCE_AUDIO)
if not reference.exists() and Path("..", REFERENCE_AUDIO).exists():
    reference = Path("..", REFERENCE_AUDIO)  # 저장소를 내려받아 notebooks 폴더에서 실행한 경우
if not reference.exists() and REFERENCE_AUDIO == "voice_sample.mp3":
    urlretrieve(SAMPLE_URL, reference)  # Colab에서는 저장소에서 바로 내려받습니다.
if not reference.exists():
    raise FileNotFoundError(f"{REFERENCE_AUDIO} 파일을 찾지 못했습니다. 파일 이름과 업로드 위치를 확인하세요.")

print("참조 음성:", reference)
display(Audio(str(reference)))

## 4. 모델 불러오기

Hugging Face에서 OmniVoice 모델(약 3.3GB)과, 참조 음성을 받아쓰는 Whisper 모델(약 1.6GB)을 내려받아 GPU에 올립니다.
처음에는 3~5분 정도 걸립니다. 한 번 불러오면 다시 실행할 필요가 없습니다.

In [ ]:
#@title 4. 모델 불러오기
import torch
from omnivoice import OmniVoice

if "model" not in globals():
    model = OmniVoice.from_pretrained(
        "k2-fsa/OmniVoice",
        device_map="cuda:0",
        dtype=torch.float16,
        load_asr=True,  # 참조 음성의 내용을 자동으로 받아쓰는 Whisper 모델도 함께 불러옵니다.
    )
print("모델을 불러왔습니다.")

## 5. 목소리 준비하기

참조 음성을 한 번 분석해서 목소리 정보를 만들어 둡니다. 참조 음성에서 말한 내용도 자동으로 받아씁니다.
이렇게 만들어 둔 목소리는 문장을 바꿔도 계속 다시 씁니다. 참조 음성을 바꿨다면 3번과 5번 셀을 다시 실행하세요.

In [ ]:
#@title 5. 목소리 준비
voice = model.create_voice_clone_prompt(ref_audio=str(reference))
print("목소리를 준비했습니다. 받아쓴 참조 문장:", voice.ref_text)

## 6. 읽을 문장 입력하기

입력 칸(`TEXT`)에 읽을 문장을 적습니다. 기본값은 professor 스킬의 예시 답변입니다.

- **Colab만 쓰는 경우:** 하고 싶은 말을 직접 적습니다. 교수님이 해 줄 법한 피드백을 2~4문장으로 적어 보세요.
- **VSCode와 에이전트를 쓰는 경우:** professor 스킬이 만든 음성용 답변을 그대로 붙여넣습니다.

처음에는 2~4문장 정도로 짧게 시작하세요. 문장이 길수록 합성 시간이 늘어납니다.

In [ ]:
#@title 6. 읽을 문장
TEXT = "일단 뭐가 제일 막히는 건지 보자. 피곤한 건지, 뭘 해야 할지 안 보이는 건지에 따라 다르잖아. 오늘은 풀고 싶은 문제하고 막힌 부분만 한 문장씩 적어 봐. 그걸 보고 다음에 뭘 할지 같이 정리하면 되지."  #@param {type:"string"}

text = " ".join(TEXT.split())
if not text:
    raise ValueError("TEXT에 읽을 문장을 적어 주세요.")
print(f"읽을 문장 ({len(text)}자): {text}")

## 7. 음성 합성하고 듣기

준비한 목소리로 6번의 문장을 읽습니다. 생성 옵션은 기본값 그대로 둡니다.

합성은 실행할 때마다 결과가 조금씩 다르고, 가끔 문장 일부(주로 첫 문장)를 건너뛸 때가 있습니다.
그래서 만든 음성을 Whisper로 다시 받아써서 입력한 문장과 비교하고, 빠진 부분이 있으면 자동으로 다시 만듭니다(최대 3번).
억양이 마음에 들지 않으면 이 셀만 다시 실행해 보세요.

In [ ]:
#@title 7. 음성 합성하고 듣기
import re
import time
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
import soundfile as sf
from IPython.display import Audio, display


def heard_ratio(expected, heard):
    """입력 문장의 글자 중 받아쓰기에서도 들린 비율입니다. 띄어쓰기와 문장 부호는 빼고 셉니다."""
    a = re.sub(r"[^0-9A-Za-z가-힣]", "", expected).lower()
    b = re.sub(r"[^0-9A-Za-z가-힣]", "", heard).lower()
    if not a:
        return 1.0
    matcher = SequenceMatcher(None, a, b, autojunk=False)
    return sum(block.size for block in matcher.get_matching_blocks()) / len(a)


started = time.time()
best = None
for attempt in range(1, 4):
    audio = model.generate(text=text, voice_clone_prompt=voice)[0]
    heard = model.transcribe((audio, model.sampling_rate))  # 만든 음성을 Whisper로 다시 받아씁니다.
    ratio = heard_ratio(text, heard)
    if best is None or ratio > best[0]:
        best = (ratio, audio, heard)
    if ratio >= 0.9:
        break
    if attempt < 3:
        print(f"문장 일부가 빠진 것 같아 다시 만듭니다. ({attempt}/3)")
ratio, audio, heard = best

output = Path(f"professor_{datetime.now():%m%d_%H%M%S}.wav")
sf.write(str(output), audio, model.sampling_rate)

print(f"완료: {time.time() - started:.1f}초 걸림 / 음성 {len(audio) / model.sampling_rate:.1f}초 / 파일 {output}")
print("받아쓰기 확인:", heard)
if ratio < 0.9:
    print("받아쓰기가 입력 문장과 조금 다릅니다. 영어 단어나 숫자는 다르게 받아써질 수 있으니 직접 들어 보고, "
          "빠진 부분이 있으면 이 셀을 다시 실행하세요.")
display(Audio(str(output)))

## 8. 파일로 받기

- **Colab:** 아래 셀이 만드는 링크를 누르면 방금 만든 WAV 파일이 내려받아집니다. 왼쪽 폴더 아이콘에서 파일을 우클릭해 받을 수도 있습니다.
- **VSCode:** 왼쪽 Colab 아이콘의 **Contents**에서 파일을 우클릭하고 **Download...**를 누릅니다. VSCode에서는 Colab의 다운로드 기능(`files.download`)이 동작하지 않습니다.

노트북을 다른 사람과 공유할 때는 실행 결과(음성 포함)를 지우고 저장하세요.

In [ ]:
#@title 8. 파일로 받기
import base64
from IPython.display import HTML, display

encoded = base64.b64encode(output.read_bytes()).decode()
display(HTML(f'<a download="{output.name}" href="data:audio/wav;base64,{encoded}">{output.name} 내려받기</a>'))
print("저장 위치:", output.resolve())

## 더 해 보기

1. 같은 문장으로 7번 셀만 여러 번 실행해 보세요. 억양이 얼마나 달라지나요?
2. 짧은 문장과 긴 문장을 각각 읽혀 보세요. 어느 쪽이 더 자연스럽게 들리나요?
3. VSCode를 쓴다면 같은 질문을 스킬 없이 물은 답변과 professor 스킬 답변으로 각각 들어 보세요.

## 막힐 때

| 증상 | 해결 |
| --- | --- |
| GPU가 연결되지 않았다는 오류 | 런타임 > 런타임 유형 변경 > T4 GPU를 고르고 모두 실행 |
| GPU를 할당받지 못했다는 안내 | 무료 GPU는 사용량이 많으면 잠시 막힐 수 있습니다. 잠시 뒤 다시 시도합니다 |
| 설치 뒤 import 오류 | 런타임 > 세션 다시 시작을 누른 뒤 모두 실행 |
| 첫 실행이 오래 걸림 | 모델(약 5GB)을 내려받는 중입니다. 두 번째부터는 빠릅니다 |
| 문장 일부가 빠지거나 어색하게 읽힘 | 7번 셀만 다시 실행합니다. 실행할 때마다 결과가 조금씩 달라집니다 |
| 메모리가 부족하다는 오류 | 문장을 줄이거나 런타임 > 세션 다시 시작 후 다시 실행 |
| 목소리를 바꿨는데 이전 목소리가 나옴 | 3번과 5번 셀을 다시 실행한 뒤 7번을 실행 |

다른 사람의 목소리를 동의 없이 복제하거나, 만든 음성을 공개하지 마세요.
OmniVoice 사용법은 [OmniVoice 저장소](https://github.com/k2-fsa/OmniVoice)와 [모델 카드](https://huggingface.co/k2-fsa/OmniVoice)를 참고하세요.